# Pandas Basics
Tren Meckel    
10/23/2024  

### Part 2: Merging Files, Adding Columns, Manipulating and Changing Dataframes
In this notebook we gain experience 
- Merging two dataframes into a single dataframe, both by adding more rows (more observations) and also by adding more columns (more variables)
- Adding new columns which are computed from existing columns
- Changing the value of columns

### Reading Multiple Files
1) Read the <TT>Copiers.csv</TT> file as we did before. <p>
2) Read the <TT>CopiersAdd.csv</TT> file as a dataframe. <p>
3) Read the <TT>CopiersService.csv</TT> file as a dataframe. <p>

These three files will be combined into a single dataframe.  Print them out so we can see what contents they contain.

In [83]:
import pandas as pd

copiers = pd.read_csv('Copiers.csv',header=2,nrows=4,index_col=0)
copiersAdd = pd.read_csv('CopiersAdded.csv',header=2,nrows=2,index_col=0)
copiersService = pd.read_csv('CopiersService.csv',index_col=0)
print(copiers)
print("\n ") # just to see 
print(copiersAdd)
print("\n ") # just to see 
print(copiersService)

                PPM    CPP Color  Count
Model                                  
Xerox 1200       42  0.120     Y  12566
Brother DP6400   20  0.072     N    340
Xerox 6500pro    93  0.045     N    710
ImageWriter DP    5  0.250     Y     37

 
              PPM  CPP Color  Count
Model                              
LaserMark 10   25  0.1     N   5905
LaserMark 20   30  0.1     N  21787

 
                Service Interval  Service Cost
Model                                         
Xerox 1200                 10000        123.50
Brother DP6400              2000        149.99
Xerox 6500pro              12000        117.45
ImageWriter DP               100        212.80
LaserMark 10                4000        134.55
LaserMark 20                5000        134.55


### Concatenating
Our first step will be to concatenate the original copiers data with the copiersAdd data.  These contain the same column labels.  Research the <TT>concat()</TT> function and apply it to create a single copiers dataset with six copiers.  

In [85]:
og_concat_with_add = pd.concat( [copiers , copiersAdd] )
print(og_concat_with_add)

                PPM    CPP Color  Count
Model                                  
Xerox 1200       42  0.120     Y  12566
Brother DP6400   20  0.072     N    340
Xerox 6500pro    93  0.045     N    710
ImageWriter DP    5  0.250     Y     37
LaserMark 10     25  0.100     N   5905
LaserMark 20     30  0.100     N  21787


### Merging
The merge operation allows us to merge two separate datasets that contain the same row labels.  Look at the copier service data -- see that it has the same row labels but two new columns.  Merge with the ammended copier data to create a single dataframe with six copiers and six columns of data.  Research the <TT>merge()</TT> command.

In [87]:
merge_copiers = pd.merge( og_concat_with_add, copiersService, how = 'outer' , on='Model') 
print( merge_copiers )

                PPM    CPP Color  Count  Service Interval  Service Cost
Model                                                                  
Brother DP6400   20  0.072     N    340              2000        149.99
ImageWriter DP    5  0.250     Y     37               100        212.80
LaserMark 10     25  0.100     N   5905              4000        134.55
LaserMark 20     30  0.100     N  21787              5000        134.55
Xerox 1200       42  0.120     Y  12566             10000        123.50
Xerox 6500pro    93  0.045     N    710             12000        117.45


### Creating a New Column
Create a new column in your copier dataframe titled 'PrintingCost'.  The values for this column are the CPP * Count.

In [91]:
merge_copiers[ 'Printing Cost' ] = merge_copiers['CPP'] * merge_copiers['Count']
print( merge_copiers ) 

                PPM    CPP Color  Count  Service Interval  Service Cost  \
Model                                                                     
Brother DP6400   20  0.072     N    340              2000        149.99   
ImageWriter DP    5  0.250     Y     37               100        212.80   
LaserMark 10     25  0.100     N   5905              4000        134.55   
LaserMark 20     30  0.100     N  21787              5000        134.55   
Xerox 1200       42  0.120     Y  12566             10000        123.50   
Xerox 6500pro    93  0.045     N    710             12000        117.45   

                Printing Cost  
Model                          
Brother DP6400          24.48  
ImageWriter DP           9.25  
LaserMark 10           590.50  
LaserMark 20          2178.70  
Xerox 1200            1507.92  
Xerox 6500pro           31.95  


### Changing Value of Column
Change the values of the 'Color' column to boolean values.  

In [95]:
merge_copiers[ 'Color' ] =  merge_copiers[ 'Color' ].map({ 'Y': True , 'N' : False })
print( merge_copiers) 

                PPM    CPP Color  Count  Service Interval  Service Cost  \
Model                                                                     
Brother DP6400   20  0.072   NaN    340              2000        149.99   
ImageWriter DP    5  0.250   NaN     37               100        212.80   
LaserMark 10     25  0.100   NaN   5905              4000        134.55   
LaserMark 20     30  0.100   NaN  21787              5000        134.55   
Xerox 1200       42  0.120   NaN  12566             10000        123.50   
Xerox 6500pro    93  0.045   NaN    710             12000        117.45   

                Printing Cost  
Model                          
Brother DP6400          24.48  
ImageWriter DP           9.25  
LaserMark 10           590.50  
LaserMark 20          2178.70  
Xerox 1200            1507.92  
Xerox 6500pro           31.95  


### Total Cost
Next we will generate a complicated report.    We want to compare the cost of ownership of our color and b/w copiers.
- Part of the cost of ownership is the Printing Cost we have just computed.
- Another part of the cost is the service cost.  The service interval indicates how often we must service the copier.  Everytime the Count reaches a multiple of the Service Interval, then we must pay a service fee.   Compute the total service fees paid for each copier.
- Compute the cost-of-ownership of each copier.
- Then compute the average cost-of-ownership of color and b/w copiers to compare them.  

In [101]:
merge_copiers['NumService'] = (merge_copiers['Count'] // merge_copiers['Service Interval']) 
merge_copiers['TotalServiceCost'] = merge_copiers['NumService'] * merge_copiers['Service Cost']
merge_copiers['CostOfOwnership'] = merge_copiers['Printing Cost'] + merge_copiers['TotalServiceCost'] 
average_cost_of_ownership = merge_copiers.groupby('Color')['CostOfOwnership'].mean().rename(index={True: 'True', False: 'False'})

print( merge_copiers )
print("\n")
print( average_cost_of_ownership ) 


                PPM    CPP Color  Count  Service Interval  Service Cost  \
Model                                                                     
Brother DP6400   20  0.072   NaN    340              2000        149.99   
ImageWriter DP    5  0.250   NaN     37               100        212.80   
LaserMark 10     25  0.100   NaN   5905              4000        134.55   
LaserMark 20     30  0.100   NaN  21787              5000        134.55   
Xerox 1200       42  0.120   NaN  12566             10000        123.50   
Xerox 6500pro    93  0.045   NaN    710             12000        117.45   

                Printing Cost  NumService  TotalServiceCost  CostOfOwnership  
Model                                                                         
Brother DP6400          24.48           0              0.00            24.48  
ImageWriter DP           9.25           0              0.00             9.25  
LaserMark 10           590.50           1            134.55           725.05  
Lase